# Challenge Overview

Build machine learning (ML) models to detect misinformation in Portuguese news articles, and
leverage interpretability and explainability methods to analyze results on the FakeNews-PT
dataset.

# 1 Model Training & Evaluation (10 points)

a) Extract TF-IDF features from the text with a maximum number of features (terms) set to 5000.
Make sure to add smoothing for out-of-vocabulary (OOV) words (idf smoothing). Define the
minimum and maximum number of documents a term must appear in as min_df=10, and the
maximum proportion of documents a term can appear in as max_df=0.9.

Imports for the project:

In [7]:
# Feature selection through Tfidfvectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Metrics and Kfold
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay, auc


# Model Selection 
from sklearn.model_selection import GridSearchCV

# Required models
from sklearn.tree import DecisionTreeClassifier 
from sklearn import tree
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans

# Data manipulation
import pandas as pd

# Visualization
import matplotlib as plt

# Numerical Computing
import numpy as np

# import spacy
import spacy
#nlp = spacy.load("pt_core_news_lg")

import nltk

In [8]:
from nltk.corpus import stopwords
PORTUGUESE_STOPWORDS = stopwords.words('portuguese')

RANDOM_STATE = 42

Train, Validation and Test datasets without preprocessing

In [9]:
# Obtain all the text documents as a pandas data frame (train, validation and test)
train_fake_news_df = pd.read_csv("../data/train.csv")
val_fake_news_df = pd.read_csv("../data/val.csv")
test_fake_news_df = pd.read_csv("../data/test.csv")

Lemmatization of the datasets

In [15]:
# Lemmatize texts
#def lemmatize_text(text):
#    doc = nlp(text)
#    lemmatized_tokens = [token.lemma_ for token in doc if not (token.is_punct or token.is_space or token.is_digit)]
#    return " ".join(lemmatized_tokens)

#train_fake_news_df['Text'] = train_fake_news_df['Text'].apply(lemmatize_text)
#val_fake_news_df['Text'] = val_fake_news_df['Text'].apply(lemmatize_text)
#test_fake_news_df['Text'] = test_fake_news_df['Text'].apply(lemmatize_text)

In [16]:
# Save lemmatized data to a csv file
#train_fake_news_df.to_csv("../data/train_lemmatized.csv", index=False, encoding="utf-8")
#val_fake_news_df.to_csv("../data/val_lemmatized.csv", index=False, encoding="utf-8")
#test_fake_news_df.to_csv("../data/test_lemmatized.csv", index=False, encoding="utf-8")

Separate Texts from their labels

In [10]:
# Separate text documents from their labels (train, validation and test)
train_news_texts = train_fake_news_df['Text']
train_news_labels = train_fake_news_df['Label']

val_news_texts = val_fake_news_df['Text']
val_news_labels = val_fake_news_df['Label']

test_news_texts = test_fake_news_df['Text']
test_news_labels = test_fake_news_df['Label']

print("News Texts:\n")
print(train_news_texts)
print("\nNews Labels:\n")
print(train_news_labels)

News Texts:

0        PJ em operação internacional de tráfico de dro...
1        O texto foi criado - numa universidade da Repú...
2        Nunca pague multas por conduzir sem carta ou s...
3        Depois de António Guterres, uma estátua de D. ...
4        O MAR, uma droga gratuita que cura pelo menos ...
                               ...                        
50582    FMI recomenda reforma para evitar "acumulação ...
50583    Governo diz que o estado vai indemnizar as vít...
50584    Homem com cancro terminal realiza último desej...
50585    Grupo Impresa tem dívida de 189,1 milhões de e...
50586    Mãe fica chocada ao perceber quem estava a seg...
Name: Text, Length: 50587, dtype: object

News Labels:

0        1
1        1
2        0
3        0
4        0
        ..
50582    1
50583    0
50584    0
50585    0
50586    0
Name: Label, Length: 50587, dtype: int64


Apply TF-IDF algorithm

In [11]:
# Characteristics of the tf_idf feature selection that were requested
MAX_FEATURES = 5000
MIN_DF = 10
MAX_DF = 0.9
SMOOTH_IDF = True
STOP_WORDS = PORTUGUESE_STOPWORDS

# Optional characteristics that improve feature selection
LOWER_CASE = True


# Tf-idf vectorizer initialization
tfidf_vectorizer = TfidfVectorizer(
    max_features= MAX_FEATURES,
    min_df= MIN_DF,
    max_df= MAX_DF,
    smooth_idf= SMOOTH_IDF,
    stop_words= STOP_WORDS,
    lowercase= LOWER_CASE,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)

In [12]:
# Obtain the matrix of df and idf frequencies for training
train_tfidf = tfidf_vectorizer.fit_transform(train_news_texts)

# Get the name of the features that were determined
feature_names = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", train_tfidf.shape)
print("Exemplo de termos:", feature_names[:20])
train_tfidf

TF-IDF matrix shape: (50587, 5000)
Exemplo de termos: ['abaixo' 'abandonado' 'abandonar' 'abandono' 'abandonou' 'abastecimento'
 'aberta' 'abertas' 'aberto' 'abertos' 'abertura' 'abordagem' 'abordar'
 'aborto' 'abrange' 'abre' 'abrigo' 'abril' 'abrir' 'abriu']


<50587x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 4153153 stored elements in Compressed Sparse Row format>

In [13]:
# Transform the validation and testing data
val_tfidf = tfidf_vectorizer.transform(val_news_texts)
test_tfidf = tfidf_vectorizer.transform(test_news_texts)

Same for the lemmatized datasets

In [14]:
train_fake_news_df_lemmatized= pd.read_csv("../data/train_lemmatized.csv")
val_fake_news_df_lemmatized = pd.read_csv("../data/val_lemmatized.csv")
test_fake_news_df_lemmatized = pd.read_csv("../data/test_lemmatized.csv")

# Separate text documents from their labels (train, validation and test)
train_news_texts_lemmatized = train_fake_news_df_lemmatized['Text']
train_news_labels_lemmatized = train_fake_news_df_lemmatized['Label']

val_news_texts_lemmatized = val_fake_news_df_lemmatized['Text']
val_news_labels_lemmatized = val_fake_news_df_lemmatized['Label']

test_news_texts_lemmatized = test_fake_news_df_lemmatized['Text']
test_news_labels_lemmatized = test_fake_news_df_lemmatized['Label']

# Tf-idf vectorizer initialization
tfidf_vectorizer_lemmatized = TfidfVectorizer(
    max_features= MAX_FEATURES,
    min_df= MIN_DF,
    max_df= MAX_DF,
    smooth_idf= SMOOTH_IDF,
    stop_words= STOP_WORDS,
    lowercase= LOWER_CASE,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)

train_tfidf_lemmatized = tfidf_vectorizer.fit_transform(train_news_texts_lemmatized)

# Transform the validation and testing data
val_tfidf_lemmatized = tfidf_vectorizer.transform(val_news_texts_lemmatized)
test_tfidf_lemmatized = tfidf_vectorizer.transform(test_news_texts_lemmatized)

b) Train the following models using 5-fold cross-validation, tune key hyperparameters systematically
(e.g., regularization strength λ, tree depth), and document your hyperparameter search process.
1) Decision Tree
2) Gaussian Naive Bayes
3) Logistic Regression with L2 regularization
4) Logistic Regression with L1 regularization
5) Multi-Layer Perceptron (MLP)

In [15]:
# KFold that will be used for All the models
K_FOLDS = 5
SHUFFLE = True

SKF = StratifiedKFold(n_splits= K_FOLDS, shuffle= SHUFFLE, random_state= RANDOM_STATE)

SCORING = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1': make_scorer(f1_score, average='weighted')
}

In [23]:
def get_df(grid_search,
           X_train, y_train,
           X_val, y_val,
           X_test, y_test) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Perform hyperparameter tuning and evaluate model performance across datasets.
    
    This function fits a GridSearchCV or RandomizedSearchCV object on the training data,
    retrieves the best estimator, evaluates it on validation and test sets, and returns
    two DataFrames containing the performance metrics.
    
    Parameters
    ----------
    grid_search : GridSearchCV or RandomizedSearchCV
        Initialized search object with estimator, parameter grid/distributions, 
        cv strategy, and scoring metrics already configured.
    X_train : array-like of shape (n_samples, n_features)
        Training feature matrix (e.g., TF-IDF vectors).
    y_train : array-like of shape (n_samples,)
        Training target labels.
    X_val : array-like of shape (n_samples, n_features)
        Validation feature matrix.
    y_val : array-like of shape (n_samples,)
        Validation target labels.
    X_test : array-like of shape (n_samples, n_features)
        Test feature matrix.
    y_test : array-like of shape (n_samples,)
        Test target labels.
    
    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        A tuple containing two DataFrames:
        - val_metrics_df: Validation set metrics (accuracy, precision, recall, f1)
        - test_metrics_df: Test set metrics (accuracy, precision, recall, f1)
    
    Side Effects
    ------------
    Prints to console:
        - Best model parameters
        - Validation set metrics
        - Test set metrics
    
    """
    # Fit the grid search
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    
    print("\nBest Parameters:\n")
    print(grid_search.best_params_, "\n")
    
    # Validation set predictions and metrics
    y_val_pred = best_model.predict(X_val)
    val_metrics = {
        'Dataset': ['Validation'],
        'Accuracy': [accuracy_score(y_val, y_val_pred)],
        'Precision': [precision_score(y_val, y_val_pred, average='weighted')],
        'Recall': [recall_score(y_val, y_val_pred, average='weighted')],
        'F1-Score': [f1_score(y_val, y_val_pred, average='weighted')]
    }
    val_metrics_df = pd.DataFrame(val_metrics)
    
    print("\nValidation metrics:\n")
    print(f"Accuracy:  {val_metrics['Accuracy'][0]:.4f}")
    print(f"Precision: {val_metrics['Precision'][0]:.4f}")
    print(f"Recall:    {val_metrics['Recall'][0]:.4f}")
    print(f"F1-Score:  {val_metrics['F1-Score'][0]:.4f}")
    
    # Test set predictions and metrics
    y_test_pred = best_model.predict(X_test)
    test_metrics = {
        'Dataset': ['Test'],
        'Accuracy': [accuracy_score(y_test, y_test_pred)],
        'Precision': [precision_score(y_test, y_test_pred, average='weighted')],
        'Recall': [recall_score(y_test, y_test_pred, average='weighted')],
        'F1-Score': [f1_score(y_test, y_test_pred, average='weighted')]
    }
    test_metrics_df = pd.DataFrame(test_metrics)
    
    print("\nTest metrics:\n")
    print(f"Accuracy:  {test_metrics['Accuracy'][0]:.4f}")
    print(f"Precision: {test_metrics['Precision'][0]:.4f}")
    print(f"Recall:    {test_metrics['Recall'][0]:.4f}")
    print(f"F1-Score:  {test_metrics['F1-Score'][0]:.4f}")
    
    return val_metrics_df, test_metrics_df

Decision Tree

In [30]:
# Define a hyperparameter tuning algorithm for decision tree
def get_best_decision_tree(max_depth: list,
                           min_samples_split: list,
                           min_samples_leaf: list,
                           max_features: int,
                           random_state: int,
                           class_weight: list,
                           criteria: str,
                           skf: StratifiedKFold,
                           scoring_dict: dict) -> GridSearchCV:

   # Param grid for Decision Tree 
   param_grid = {
      'max_depth': max_depth,  
      'min_samples_split': min_samples_split,     
      'min_samples_leaf': min_samples_leaf,
      'class_weight': class_weight  
   }
    
   # Initialize Decision Tree Classifier
   decision_tree_model = DecisionTreeClassifier(
      max_features=max_features,
      random_state=random_state
   )
   
   # Search for the best set of parameters that optimize the model based on criteria
   best_decision_tree_model = GridSearchCV(
      estimator=decision_tree_model,
      param_grid=param_grid,
      cv=skf,
      scoring=scoring_dict,
      refit=criteria,
      n_jobs=-1  # Add this for parallel processing
   )
   
   return best_decision_tree_model

In [41]:
# Decision Tree
    
# Parameters
max_depth = [10, 30, 50, 70]
min_samples_split = [2, 5, 10, 20]
min_samples_leaf = [1, 2, 4]
class_weight = [None, 'balanced']

CRITERIA = "f1"

best_decision_tree_model = get_best_decision_tree(
    max_depth = max_depth,
    min_samples_split = min_samples_split,
    min_samples_leaf = min_samples_leaf,
    max_features = MAX_FEATURES,
    random_state = RANDOM_STATE,
    class_weight = class_weight,
    criteria = CRITERIA ,
    skf = SKF,
    scoring_dict=SCORING
)

results_df = get_df(
    grid_search=best_decision_tree_model,
    X_train=train_tfidf,
    y_train=train_news_labels,
    X_val=val_tfidf,
    y_val=val_news_labels,
    X_test=test_tfidf,
    y_test=test_news_labels
)

results_df


Best Parameters:

{'class_weight': 'balanced', 'max_depth': 50, 'min_samples_leaf': 1, 'min_samples_split': 2} 


Validation metrics:

Accuracy:  0.8287
Precision: 0.8293
Recall:    0.8287
F1-Score:  0.8287

Test metrics:

Accuracy:  0.8305
Precision: 0.8307
Recall:    0.8305
F1-Score:  0.8304


(      Dataset  Accuracy  Precision    Recall  F1-Score
 0  Validation  0.828748   0.829342  0.828748    0.8287,
   Dataset  Accuracy  Precision    Recall  F1-Score
 0    Test  0.830487   0.830744  0.830487  0.830447)

Guassian Naive Bayes

In [16]:
def get_gaussian_NB_df(var_smoothing_values:list,
                       X_train,
                       y_train,
                       X_val,
                       y_val,
                       X_test,
                       y_test):
    # Store results for each parameter value
    results = []

    print("Tuning var_smoothing parameter for Gaussian Naive Bayes...\n")

    for var_smooth in var_smoothing_values:
        # Initialize and train model
        model = GaussianNB(var_smoothing=var_smooth)
        model.fit(X_train, y_train)
        
        # Validation predictions and metrics
        y_val_pred = model.predict(X_val)
        val_f1 = f1_score(y_val, y_val_pred, average='weighted')
        
        results.append({
            'var_smoothing': var_smooth,
            'val_f1_score': val_f1,
            'model': model
        })
        
        print(f"var_smoothing={var_smooth:.0e} | Validation F1-Score: {val_f1:.4f}")

    # Find best model based on validation F1-score
    best_result = max(results, key=lambda x: x['val_f1_score'])
    best_model = best_result['model']


    print(f"Best Parameter: var_smoothing = {best_result['var_smoothing']:.0e}")
    print(f"Best Validation F1-Score: {best_result['val_f1_score']:.4f}")
   

    # Evaluate best model on validation and test sets
    y_val_pred = best_model.predict(X_val)
    val_metrics = {
        'Dataset': ['Validation'],
        'Accuracy': [accuracy_score(y_val, y_val_pred)],
        'Precision': [precision_score(y_val, y_val_pred, average='weighted')],
        'Recall': [recall_score(y_val, y_val_pred, average='weighted')],
        'F1-Score': [f1_score(y_val, y_val_pred, average='weighted')]
    }
    val_metrics_df = pd.DataFrame(val_metrics)

    print("\nValidation Metrics:\n")
    print(f"Accuracy:  {val_metrics['Accuracy'][0]:.4f}")
    print(f"Precision: {val_metrics['Precision'][0]:.4f}")
    print(f"Recall:    {val_metrics['Recall'][0]:.4f}")
    print(f"F1-Score:  {val_metrics['F1-Score'][0]:.4f}")

    # Test set evaluation
    y_test_pred = best_model.predict(X_test)
    test_metrics = {
        'Dataset': ['Test'],
        'Accuracy': [accuracy_score(y_test, y_test_pred)],
        'Precision': [precision_score(y_test, y_test_pred, average='weighted')],
        'Recall': [recall_score(y_test, y_test_pred, average='weighted')],
        'F1-Score': [f1_score(y_test, y_test_pred, average='weighted')]
    }
    test_metrics_df = pd.DataFrame(test_metrics)

    print("\nTest Metrics:\n")
    print(f"Accuracy:  {test_metrics['Accuracy'][0]:.4f}")
    print(f"Precision: {test_metrics['Precision'][0]:.4f}")
    print(f"Recall:    {test_metrics['Recall'][0]:.4f}")
    print(f"F1-Score:  {test_metrics['F1-Score'][0]:.4f}")

    # Combine results
    results_df = (val_metrics_df, test_metrics_df)
    results_df

In [17]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Gaussian Naive Bayes - Manual hyperparameter tuning
VAR_SMOTHING_VALUES = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

# Convert sparse matrices to dense once (outside the loop)
X_train_dense = train_tfidf.toarray()
X_val_dense = val_tfidf.toarray()
X_test_dense = test_tfidf.toarray()

results_df = get_gaussian_NB_df(
    var_smoothing_values=VAR_SMOTHING_VALUES,
    X_train=train_tfidf.toarray(),
    y_train=train_news_labels,
    X_val=val_tfidf.toarray(),
    y_val=val_news_labels,
    X_test=test_tfidf.toarray(),
    y_test=test_news_labels
)

results_df

Tuning var_smoothing parameter for Gaussian Naive Bayes...

var_smoothing=1e-09 | Validation F1-Score: 0.8432
var_smoothing=1e-08 | Validation F1-Score: 0.8432
var_smoothing=1e-07 | Validation F1-Score: 0.8432
var_smoothing=1e-06 | Validation F1-Score: 0.8431
var_smoothing=1e-05 | Validation F1-Score: 0.8431
var_smoothing=1e-04 | Validation F1-Score: 0.8431
var_smoothing=1e-03 | Validation F1-Score: 0.8420
var_smoothing=1e-02 | Validation F1-Score: 0.8399
var_smoothing=1e-01 | Validation F1-Score: 0.8305
var_smoothing=1e+00 | Validation F1-Score: 0.8055
Best Parameter: var_smoothing = 1e-09
Best Validation F1-Score: 0.8432

Validation Metrics:

Accuracy:  0.8433
Precision: 0.8436
Recall:    0.8433
F1-Score:  0.8432

Test Metrics:

Accuracy:  0.8493
Precision: 0.8500
Recall:    0.8493
F1-Score:  0.8492


In [27]:
# Do the the same for the lemmantized datasets
results_df = get_gaussian_NB_df(
    var_smoothing_values=VAR_SMOTHING_VALUES,
    X_train=train_tfidf_lemmatized.toarray(),
    y_train=train_news_labels,
    X_val=val_tfidf_lemmatized.toarray(),
    y_val=val_news_labels,
    X_test=test_tfidf_lemmatized.toarray(),
    y_test=test_news_labels
)

results_df

Tuning var_smoothing parameter for Gaussian Naive Bayes...

var_smoothing=1e-09 | Validation F1-Score: 0.8434
var_smoothing=1e-08 | Validation F1-Score: 0.8434
var_smoothing=1e-07 | Validation F1-Score: 0.8434
var_smoothing=1e-06 | Validation F1-Score: 0.8434
var_smoothing=1e-05 | Validation F1-Score: 0.8432
var_smoothing=1e-04 | Validation F1-Score: 0.8432
var_smoothing=1e-03 | Validation F1-Score: 0.8421
var_smoothing=1e-02 | Validation F1-Score: 0.8404
var_smoothing=1e-01 | Validation F1-Score: 0.8299
var_smoothing=1e+00 | Validation F1-Score: 0.8060
Best Parameter: var_smoothing = 1e-09
Best Validation F1-Score: 0.8434

Validation Metrics:

Accuracy:  0.8435
Precision: 0.8437
Recall:    0.8435
F1-Score:  0.8434

Test Metrics:

Accuracy:  0.8495
Precision: 0.8502
Recall:    0.8495
F1-Score:  0.8494


Logistic Regression with L2 Regularization

In [18]:
# Define a Generic hipertuning algorithm for logistic_regression
def get_the_best_logistic_regression(inverse_regularization_strength: list,
                                      solvers: list,
                                      max_iter: list,
                                      class_weight: list,
                                      criteria: str,
                                      verbose: int,
                                      return_train_score: bool,
                                      penalty: str,
                                      random_state: int,
                                      skf: StratifiedKFold,
                                      scoring_dict: dict) -> GridSearchCV:
    """
    Create a GridSearchCV object for hyperparameter tuning of Logistic Regression.
    
    This function configures a GridSearchCV with specified hyperparameter ranges
    for Logistic Regression, enabling exhaustive search over the parameter space
    to find the optimal model configuration based on multiple scoring metrics.
    
    Parameters
    ----------
    inverse_regularization_strength : list
        List of C values (inverse of regularization strength). Smaller values 
        specify stronger regularization. Example: [0.001, 0.01, 0.1, 1, 10, 100].
    solvers : list
        List of optimization algorithms to try. Options include:
        - 'liblinear': Good for small datasets, supports L1 and L2
        - 'lbfgs': Memory-efficient, supports only L2
        - 'saga': Supports L1, L2, and elastic net, good for large datasets
        - 'newton-cg', 'sag': Other available solvers
        Example: ['liblinear', 'saga'] for L1, ['lbfgs', 'liblinear', 'saga'] for L2.
    max_iter : list
        List of maximum iteration counts for solver convergence.
        Example: [100, 200, 500, 1000].
    class_weight : list
        List of class weighting strategies. Options:
        - None: All classes have equal weight
        - 'balanced': Automatically adjusts weights inversely proportional to class frequencies
        Example: [None, 'balanced'].
    criteria : str
        Scoring metric to optimize and use for selecting the best model (refit metric).
        Must be one of the keys in scoring_dict (e.g., 'f1', 'accuracy', 
        'precision', 'recall').
    verbose : int
        Controls the verbosity of GridSearchCV output.
        - 0: Silent
        - 1: Progress for each fold
        - 2: Progress for each parameter combination and fold (recommended)
        - 3+: More detailed output
    return_train_score : bool
        Whether to include training scores in cv_results_.
        Set to True to diagnose overfitting; False to save memory and time.
    penalty : str
        Type of regularization penalty. Options:
        - 'l1': Lasso regularization (feature selection, sparse solutions)
        - 'l2': Ridge regularization (default, shrinks coefficients)
        - 'elasticnet': Combination of L1 and L2
        - None: No regularization
    random_state : int
        Random seed for reproducibility of results.
    skf : StratifiedKFold
        Cross-validation splitter that preserves class distribution in each fold.
        Should be pre-configured with n_splits, shuffle, and random_state.
    scoring_dict : dict
        Dictionary mapping metric names to sklearn scorer objects.
        Example: {
            'accuracy': make_scorer(accuracy_score),
            'precision': make_scorer(precision_score, average='weighted'),
            'recall': make_scorer(recall_score, average='weighted'),
            'f1': make_scorer(f1_score, average='weighted')
        }
        All metrics will be computed during cross-validation.
    
    Returns
    -------
    GridSearchCV
        Configured but unfitted GridSearchCV object ready for training.
        Call .fit(X_train, y_train) to execute the hyperparameter search.
    
    Notes
    -----
    - For L1 penalty, only 'liblinear' and 'saga' solvers are compatible.
    - For L2 penalty, all solvers are compatible.
    - The returned GridSearchCV object must be fitted before use.
    - The best model will be selected based on the 'criteria' metric.
    - All metrics in scoring_dict will be computed and stored in cv_results_.
    - Consider adding n_jobs=-1 to GridSearchCV for parallel processing.
    
    See Also
    --------
    sklearn.linear_model.LogisticRegression : The underlying estimator
    sklearn.model_selection.GridSearchCV : The search strategy used
    get_df : Function to evaluate the fitted grid_search and get results
    """
    # Param grid for Logistic Regression
    param_grid = {
        'C': inverse_regularization_strength,  
        'solver': solvers,     
        'max_iter': max_iter,            
        'class_weight': class_weight          
    }
    
    # Initialize Logistic Regression with specified penalty
    logistic_regression_model = LogisticRegression(
        penalty=penalty,
        random_state=random_state
    )
    
    # Search for the best set of parameters that optimize the model based on criteria
    grid_search = GridSearchCV(
        estimator=logistic_regression_model,
        param_grid=param_grid,
        cv=skf,
        scoring=scoring_dict,
        refit=criteria,
        verbose=verbose,
        return_train_score=return_train_score,
        n_jobs=-1  # Add this for parallel processing
    )
    
    return grid_search



In [19]:
def get_df(grid_search,
           X_train, y_train,
           X_val, y_val,
           X_test, y_test) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Perform hyperparameter tuning and evaluate model performance across datasets.
    
    This function fits a GridSearchCV or RandomizedSearchCV object on the training data,
    retrieves the best estimator, evaluates it on validation and test sets, and returns
    two DataFrames containing the performance metrics.
    
    Parameters
    ----------
    grid_search : GridSearchCV or RandomizedSearchCV
        Initialized search object with estimator, parameter grid/distributions, 
        cv strategy, and scoring metrics already configured.
    X_train : array-like of shape (n_samples, n_features)
        Training feature matrix (e.g., TF-IDF vectors).
    y_train : array-like of shape (n_samples,)
        Training target labels.
    X_val : array-like of shape (n_samples, n_features)
        Validation feature matrix.
    y_val : array-like of shape (n_samples,)
        Validation target labels.
    X_test : array-like of shape (n_samples, n_features)
        Test feature matrix.
    y_test : array-like of shape (n_samples,)
        Test target labels.
    
    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        A tuple containing two DataFrames:
        - val_metrics_df: Validation set metrics (accuracy, precision, recall, f1)
        - test_metrics_df: Test set metrics (accuracy, precision, recall, f1)
    
    Side Effects
    ------------
    Prints to console:
        - Best model parameters
        - Validation set metrics
        - Test set metrics
    
    """
    # Fit the grid search
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    
    print("\nBest Parameters:\n")
    print(grid_search.best_params_, "\n")
    
    # Validation set predictions and metrics
    y_val_pred = best_model.predict(X_val)
    val_metrics = {
        'Dataset': ['Validation'],
        'Accuracy': [accuracy_score(y_val, y_val_pred)],
        'Precision': [precision_score(y_val, y_val_pred, average='weighted')],
        'Recall': [recall_score(y_val, y_val_pred, average='weighted')],
        'F1-Score': [f1_score(y_val, y_val_pred, average='weighted')]
    }
    val_metrics_df = pd.DataFrame(val_metrics)
    
    print("\nValidation metrics:\n")
    print(f"Accuracy:  {val_metrics['Accuracy'][0]:.4f}")
    print(f"Precision: {val_metrics['Precision'][0]:.4f}")
    print(f"Recall:    {val_metrics['Recall'][0]:.4f}")
    print(f"F1-Score:  {val_metrics['F1-Score'][0]:.4f}")
    
    # Test set predictions and metrics
    y_test_pred = best_model.predict(X_test)
    test_metrics = {
        'Dataset': ['Test'],
        'Accuracy': [accuracy_score(y_test, y_test_pred)],
        'Precision': [precision_score(y_test, y_test_pred, average='weighted')],
        'Recall': [recall_score(y_test, y_test_pred, average='weighted')],
        'F1-Score': [f1_score(y_test, y_test_pred, average='weighted')]
    }
    test_metrics_df = pd.DataFrame(test_metrics)
    
    print("\nTest metrics:\n")
    print(f"Accuracy:  {test_metrics['Accuracy'][0]:.4f}")
    print(f"Precision: {test_metrics['Precision'][0]:.4f}")
    print(f"Recall:    {test_metrics['Recall'][0]:.4f}")
    print(f"F1-Score:  {test_metrics['F1-Score'][0]:.4f}")
    
    return val_metrics_df, test_metrics_df

In [ ]:
# Logistic Regression with L2 regularization

# Parameters
PENALTY = "l2"

INVERSE_REGULARIZATION_STRENGTH = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
SOLVERS = ['lbfgs', 'liblinear', 'saga']
MAX_ITER = [100, 200, 500, 1000]
CLASS_WEIGHT = [None, 'balanced']

CRITERIA = "f1"
VERBOSE = 2
RETURN_TRAIN_SCORE = False 

grid_search = get_the_best_logistic_regression(
    inverse_regularization_strength = INVERSE_REGULARIZATION_STRENGTH,
    solvers = SOLVERS,
    max_iter = MAX_ITER,
    class_weight = CLASS_WEIGHT,
    criteria = CRITERIA ,
    verbose = VERBOSE,
    return_train_score = RETURN_TRAIN_SCORE,
    penalty = PENALTY,
    random_state = RANDOM_STATE,
    skf = SKF,
    scoring_dict=SCORING
)

results_df = get_df(
    grid_search=grid_search,
    X_train=train_tfidf,
    y_train=train_news_labels,
    X_val=val_tfidf,
    y_val=val_news_labels,
    X_test=test_tfidf,
    y_test=test_news_labels
)

results_df



Fitting 5 folds for each of 168 candidates, totalling 840 fits


/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else s

[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.3s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solve

/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else s

[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=saga; total time=   1.0s
[CV] END C=0.001, class_weight=None, max_iter=500, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=saga; total time=   1.0s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=saga; total time=   1.1s
[CV] END C=0.001, class_weight=None, max_iter=500, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=saga; total time=   1.0s
[CV] END C=0.001, class_weight=None, max_iter=500, solver=lb

...

/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END C=1000, class_weight=balanced, max_iter=1000, solver=saga; total time=  36.4s
[CV] END C=1000, class_weight=balanced, max_iter=1000, solver=saga; total time=  36.3s


/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


[CV] END C=1000, class_weight=balanced, max_iter=1000, solver=saga; total time=  36.1s

Best Parameters:

{'C': 10, 'class_weight': 'balanced', 'max_iter': 100, 'solver': 'lbfgs'} 


Validation metrics:

Accuracy:  0.8925
Precision: 0.8925
Recall:    0.8925
F1-Score:  0.8925

Test metrics:

Accuracy:  0.8945
Precision: 0.8945
Recall:    0.8945
F1-Score:  0.8945


(      Dataset  Accuracy  Precision    Recall  F1-Score
 0  Validation  0.892473   0.892501  0.892473  0.892474,
   Dataset  Accuracy  Precision    Recall  F1-Score
 0    Test  0.894529   0.894548  0.894529  0.894527)

In [21]:
# Do the same for lemantized datasets

results_df = get_df(
    grid_search=grid_search,
    X_train=train_tfidf_lemmatized,
    y_train=train_news_labels,
    X_val=val_tfidf_lemmatized,
    y_val=val_news_labels,
    X_test=test_tfidf_lemmatized,
    y_test=test_news_labels
)
results_df

Fitting 5 folds for each of 168 candidates, totalling 840 fits
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=lbfgs; total time=   0.1s


/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else s

[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.3s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.3s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.3s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=100, solver=liblinear; total time=   0.4s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=lbfgs; total time=   0.1s


/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else s

[CV] END C=0.001, class_weight=None, max_iter=200, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.2s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=200, solver=liblinear; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=500, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=500, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=500, solver=lbfgs; total time=   0.1s
[CV] END C=0.001, class_weight=None, max_iter=500, solver=lbfgs; total time=   0.0s
[CV] END C=0.001, class_weight=None, max_iter=500, solver=lbfgs; total time=   0.1s


...

/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END C=1000, class_weight=balanced, max_iter=1000, solver=saga; total time=  39.5s
[CV] END C=1000, class_weight=balanced, max_iter=1000, solver=saga; total time=  39.8s
[CV] END C=1000, class_weight=balanced, max_iter=1000, solver=saga; total time=  39.6s


/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/emaferrao/Projects/NTT_Data_merit_challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


[CV] END C=1000, class_weight=balanced, max_iter=1000, solver=saga; total time=  39.4s

Best Parameters:

{'C': 10, 'class_weight': 'balanced', 'max_iter': 100, 'solver': 'lbfgs'} 



ValueError: Found input variables with inconsistent numbers of samples: [6324, 50587]

Logistic Regression with L1 regularization

In [ ]:
# Logistic Regression with L1 regularization
# Parameters
PENALTY = "l1"

INVERSE_REGULARIZATION_STRENGTH = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
SOLVERS = ['liblinear', 'saga'] 
MAX_ITER = [100, 200, 500, 1000]
CLASS_WEIGHT = [None, 'balanced']

CRITERIA = "f1"
VERBOSE = 2
RETURN_TRAIN_SCORE = False 

grid_search = get_the_best_logistic_regression(
    inverse_regularization_strength = INVERSE_REGULARIZATION_STRENGTH,
    solvers = SOLVERS,
    max_iter = MAX_ITER,
    class_weight = CLASS_WEIGHT,
    criteria = CRITERIA ,
    verbose = VERBOSE,
    return_train_score = RETURN_TRAIN_SCORE,
    penalty = PENALTY,
    random_state = RANDOM_STATE,
    skf = SKF,
    scoring_dict=SCORING
)

results_df = get_df(
    grid_search=grid_search,
    X_train=train_tfidf,
    y_train=train_news_labels,
    X_val=val_tfidf,
    y_val=val_news_labels,
    X_test=test_tfidf,
    y_test=test_news_labels
)

results_df

In [ ]:
# Do the same for lemantized datasets
results_df = get_df(
    grid_search=grid_search,
    X_train=train_tfidf_lemmatized,
    y_train=train_news_labels,
    X_val=val_tfidf_lemmatized,
    y_val=val_news_labels,
    X_test=test_tfidf_lemmatized,
    y_test=test_news_labels
)
results_df

Multi-layer Perceptron (MLP)

c) Create a comparison table with test metrics: Accuracy, Precision, Recall, and F1-score. For the
best classifier, draw its ROC curve and compute AUC.

# 2 Model Interpretation (7 points)

a) For your best Logistic Regression model, extract and visualize the weights in a bar plot:
1) Top 10 words most indicative of fake news
2) Top 10 words most indicative of real news

b) Compare L1 vs L2 regularized models: How many features have non-zero weights in each? What
does this tell you about feature selection? When would you prefer L1 vs L2 regularization for
text classification?

c) For your best Logistic Regression model, select samples in the validation set with ID 2921,
2437, 5557, 1697, and extract explanations with LIME (Ribeiro et al., 2016; Lundberg and
Lee, 2017).

d) For your MLP, select samples in the validation set with ID 2921, 2437, 5557, 1697, and
extract explanations with LIME and permutation importance. For permutation importance,
select 1K random samples. Visualize the results and discuss their differences.

# 3 Clustering (3 points)

a) Apply K-Means with K=5 on your training set.

b) Inspect 3 documents closest to each centroid, and afterwards, assign semantic labels to each
cluster (e.g., “political fake news”, “health misinformation”).

c) Visualize clusters in 2D using PCA. Create two plots: one colored by cluster assignment, one by
true label.